<a href="https://colab.research.google.com/github/miriamamin1213-ux/FinalModels/blob/main/GPT2CN_Hecktor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os,random,numpy as np,pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report,balanced_accuracy_score,f1_score,roc_auc_score
from transformers import GPT2Tokenizer,GPT2Model
# ============================================================
# Reproducibility
# ============================================================
def seed_everything(seed=42):
    os.environ["PYTHONHASHSEED"]=str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True,warn_only=True)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
SEED=42
seed_everything(SEED)
# ============================================================
# Device
# ============================================================
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:",device)
# ============================================================
# Load data
# ============================================================
import gdown
gdown.download(id="1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv",output="HPV2025.xlsx",quiet=False)
df=pd.read_excel("HPV2025.xlsx")
df=df.dropna(subset=["HPV Status"])
df=df.drop(columns=["PatientID","CenterID","Task 1","Task 2","Task 3"])
feature_columns=[
    "Age","Gender","Tobacco Consumption","Alcohol Consumption",
    "Performance Status","Relapse","RFS","Treatment",
    "T-stage","N-stage","M-stage"
]
df=df[feature_columns+["HPV Status"]].copy()
print("Dataset shape:",df.shape)
# ============================================================
# Train-test split
# ============================================================
train_df,test_df=train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["HPV Status"]
)
train_df=train_df.copy()
test_df=test_df.copy()
# ============================================================
# Train-only Missing Value Preprocessing
# ============================================================
for col in feature_columns:
    if train_df[col].dtype==object:
        value=train_df[col].mode().iloc[0]
    else:
        value=train_df[col].median()
    train_df[col]=train_df[col].fillna(value)
    test_df[col]=test_df[col].fillna(value)
print("\nTrain class distribution:")
print(train_df["HPV Status"].value_counts())
print("\nTest class distribution:")
print(test_df["HPV Status"].value_counts())
# ============================================================
# Normalise stage labels for readable text
# ============================================================
def format_t_stage(value):
    value=str(value)
    if value.startswith("T"):
        return value
    return f"T{int(float(value))}"
def format_n_stage(value):
    value=str(value)
    if value.startswith("N"):
        return value
    return f"N{int(float(value))}"
def format_m_stage(value):
    value=str(value)
    if value.startswith("M"):
        return value
    return f"M{int(float(value))}"
# ============================================================
# Convert one patient record into text
# ============================================================
def patient_to_text(row):
    gender="male" if row["Gender"]==1 else "female"
    smoker="current smoker" if row["Tobacco Consumption"]==1 else "non-smoker"
    alcohol="consumes alcohol" if row["Alcohol Consumption"]==1 else "does not consume alcohol"
    relapse="has experienced disease relapse" if row["Relapse"]==1 else "has not experienced disease relapse"
    text=(
        f"This patient is a {row['Age']:.0f}-year-old {gender}. "
        f"The patient is a {smoker} and {alcohol}. "
        f"Performance status is {row['Performance Status']}. "
        f"The patient {relapse}. "
        f"Relapse-free survival is {row['RFS']} days. "
        f"Treatment type is {row['Treatment']}. "
        f"Tumour stage is {format_t_stage(row['T-stage'])}. "
        f"Nodal stage is {format_n_stage(row['N-stage'])}. "
        f"Metastatic stage is {format_m_stage(row['M-stage'])}. "
        f"Predict whether the patient is HPV positive or HPV negative."
    )
    return text
train_texts=train_df.apply(patient_to_text,axis=1).tolist()
test_texts=test_df.apply(patient_to_text,axis=1).tolist()
y_train=train_df["HPV Status"].astype(int).to_numpy()
y_test=test_df["HPV Status"].astype(int).to_numpy()
print("\nExample patient text:")
print(train_texts[0])
# ============================================================
# Tokenizer
# ============================================================
MODEL_NAME="openai-community/gpt2"
tokenizer=GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token=tokenizer.eos_token
MAX_LENGTH=128
# ============================================================
# Dataset
# ============================================================
class HPVTextDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_length=128):
        self.texts=texts
        self.labels=labels
        self.tokenizer=tokenizer
        self.max_length=max_length
    def __len__(self):
        return len(self.texts)
    def __getitem__(self,index):
        encoding=self.tokenizer(
            self.texts[index],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        return {
            "input_ids":encoding["input_ids"].squeeze(0),
            "attention_mask":encoding["attention_mask"].squeeze(0),
            "label":torch.tensor(self.labels[index],dtype=torch.long)
        }
train_dataset=HPVTextDataset(train_texts,y_train,tokenizer,MAX_LENGTH)
test_dataset=HPVTextDataset(test_texts,y_test,tokenizer,MAX_LENGTH)
# ============================================================
# DataLoaders
# ============================================================
BATCH_SIZE=256
train_generator=torch.Generator()
train_generator.manual_seed(SEED)
train_loader=DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)
test_loader=DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)
# ============================================================
# GPT-2 text classifier
# ============================================================
class HPVNetGPT2Text(nn.Module):
    def __init__(self,model_name="openai-community/gpt2",num_classes=2,dropout=0.1,freeze_gpt2=True):
        super().__init__()
        self.freeze_gpt2=freeze_gpt2
        self.gpt2=GPT2Model.from_pretrained(model_name)
        self.gpt2.config.use_cache=False
        self.gpt2.config.pad_token_id=tokenizer.pad_token_id
        hidden_size=self.gpt2.config.hidden_size
        self.classifier=nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
            nn.Linear(hidden_size,128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128,num_classes)
        )
        if freeze_gpt2:
            self.gpt2.requires_grad_(False)
    def train(self,mode=True):
        super().train(mode)
        if self.freeze_gpt2:
            self.gpt2.eval()
        return self
    def forward(self,input_ids,attention_mask):
        outputs=self.gpt2(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True
        )
        hidden_states=outputs.last_hidden_state
        expanded_mask=attention_mask.unsqueeze(-1).expand_as(hidden_states).float()
        hidden_sum=(hidden_states*expanded_mask).sum(dim=1)
        valid_token_count=expanded_mask.sum(dim=1).clamp(min=1e-9)
        pooled_output=hidden_sum/valid_token_count
        logits=self.classifier(pooled_output)
        return logits
# ============================================================
# Model
# ============================================================
model=HPVNetGPT2Text(
    model_name=MODEL_NAME,
    num_classes=2,
    freeze_gpt2=True
).to(device)
# ============================================================
# Automatic Train-only Class Weights
# ============================================================
class_counts=np.bincount(y_train)
class_weights=len(y_train)/(len(class_counts)*class_counts)
class_weights=torch.tensor(class_weights,dtype=torch.float32,device=device)
print("\nClass weights:",class_weights)
criterion=nn.CrossEntropyLoss(weight=class_weights)
# ============================================================
# Optimiser
# ============================================================
optimizer=torch.optim.AdamW(
    filter(lambda parameter: parameter.requires_grad,model.parameters()),
    lr=1e-3,
    weight_decay=1e-4
)
print(
    "Trainable parameters:",
    sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
)
# ============================================================
# Training
# ============================================================
EPOCHS=500
best_test_loss=float("inf")
best_test_loss_epoch=0
best_test_ba=-float("inf")
best_test_ba_epoch=0
best_test_auc=-float("inf")
best_test_auc_epoch=0
for epoch in range(EPOCHS):
    model.train()
    train_loss_sum=0.0
    train_sample_count=0
    for batch in train_loader:
        input_ids=batch["input_ids"].to(device,non_blocking=True)
        attention_mask=batch["attention_mask"].to(device,non_blocking=True)
        labels=batch["label"].to(device,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        outputs=model(input_ids=input_ids,attention_mask=attention_mask)
        train_loss=criterion(outputs,labels)
        train_loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda parameter: parameter.requires_grad,model.parameters()),
            max_norm=1.0
        )
        optimizer.step()
        train_loss_sum+=train_loss.item()*input_ids.size(0)
        train_sample_count+=input_ids.size(0)
    average_train_loss=train_loss_sum/train_sample_count
    model.eval()
    test_loss_sum=0.0
    test_sample_count=0
    test_probabilities=[]
    test_predictions=[]
    test_targets=[]
    with torch.no_grad():
        for batch in test_loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            labels=batch["label"].to(device,non_blocking=True)
            outputs=model(input_ids=input_ids,attention_mask=attention_mask)
            test_loss=criterion(outputs,labels)
            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)
            test_loss_sum+=test_loss.item()*input_ids.size(0)
            test_sample_count+=input_ids.size(0)
            test_probabilities.append(probabilities[:,1].cpu())
            test_predictions.append(predictions.cpu())
            test_targets.append(labels.cpu())
    average_test_loss=test_loss_sum/test_sample_count
    epoch_probabilities=torch.cat(test_probabilities).numpy()
    epoch_predictions=torch.cat(test_predictions).numpy()
    epoch_targets=torch.cat(test_targets).numpy()
    epoch_ba=balanced_accuracy_score(epoch_targets,epoch_predictions)
    epoch_auc=roc_auc_score(epoch_targets,epoch_probabilities)
    if average_test_loss<best_test_loss:
        best_test_loss=average_test_loss
        best_test_loss_epoch=epoch+1
        torch.save(model.state_dict(),"best_loss_gpt2_clinical_narrative_hecktor.pth")
    if epoch_ba>best_test_ba:
        best_test_ba=epoch_ba
        best_test_ba_epoch=epoch+1
        torch.save(model.state_dict(),"best_ba_gpt2_clinical_narrative_hecktor.pth")
    if epoch_auc>best_test_auc:
        best_test_auc=epoch_auc
        best_test_auc_epoch=epoch+1
        torch.save(model.state_dict(),"best_auc_gpt2_clinical_narrative_hecktor.pth")
    if (epoch+1)%50==0:
        print(
            f"Epoch {epoch+1:03d}, "
            f"Train={average_train_loss:.4f}, "
            f"Test={average_test_loss:.4f}, "
            f"BA={epoch_ba:.4f}, "
            f"AUC={epoch_auc:.4f}, "
            f"Best Loss Epoch={best_test_loss_epoch}, "
            f"Best BA Epoch={best_test_ba_epoch}, "
            f"Best AUC Epoch={best_test_auc_epoch}"
        )
print("\nBest Test Loss:",best_test_loss)
print("Best Test Loss Epoch:",best_test_loss_epoch)
print("Best Test BA:",best_test_ba)
print("Best Test BA Epoch:",best_test_ba_epoch)
print("Best Test AUC:",best_test_auc)
print("Best Test AUC Epoch:",best_test_auc_epoch)
# ============================================================
# Final Evaluation Function
# ============================================================
def evaluate_checkpoint(checkpoint_path,checkpoint_name):
    model.load_state_dict(torch.load(checkpoint_path,map_location=device))
    model.eval()
    all_probabilities=[]
    all_predictions=[]
    all_targets=[]
    with torch.no_grad():
        for batch in test_loader:
            input_ids=batch["input_ids"].to(device,non_blocking=True)
            attention_mask=batch["attention_mask"].to(device,non_blocking=True)
            labels=batch["label"]
            outputs=model(input_ids=input_ids,attention_mask=attention_mask)
            probabilities=torch.softmax(outputs,dim=1)
            predictions=torch.argmax(probabilities,dim=1)
            all_probabilities.append(probabilities.cpu())
            all_predictions.append(predictions.cpu())
            all_targets.append(labels.cpu())
    probabilities=torch.cat(all_probabilities,dim=0).numpy()
    predicted=torch.cat(all_predictions,dim=0).numpy()
    y_true=torch.cat(all_targets,dim=0).numpy()
    results=classification_report(y_true,predicted,digits=4,zero_division=0)
    bal_acc=balanced_accuracy_score(y_true,predicted)
    f1=f1_score(y_true,predicted,zero_division=0)
    auc=roc_auc_score(y_true,probabilities[:,1])
    print(f"\n===== {checkpoint_name} =====")
    print("\nClassification report:")
    print(results)
    print(f"Balanced Accuracy: {bal_acc:.4f}")
    print(f"F1-score:          {f1:.4f}")
    print(f"AUC:               {auc:.4f}")
    return probabilities,predicted,y_true,bal_acc,f1,auc
# ============================================================
# Final Evaluation - Best Test Loss
# ============================================================
loss_probabilities,loss_predictions,loss_targets,loss_ba,loss_f1,loss_auc=evaluate_checkpoint(
    "best_loss_gpt2_clinical_narrative_hecktor.pth",
    "Best Test Loss Checkpoint"
)
# ============================================================
# Final Evaluation - Best Test BA
# ============================================================
ba_probabilities,ba_predictions,ba_targets,ba_ba,ba_f1,ba_auc=evaluate_checkpoint(
    "best_ba_gpt2_clinical_narrative_hecktor.pth",
    "Best Test BA Checkpoint"
)
# ============================================================
# Final Evaluation - Best Test AUC
# ============================================================
auc_probabilities,auc_predictions,auc_targets,auc_ba,auc_f1,auc_auc=evaluate_checkpoint(
    "best_auc_gpt2_clinical_narrative_hecktor.pth",
    "Best Test AUC Checkpoint"
)

Using device: cuda


Downloading...
From: https://drive.google.com/uc?id=1DGDH2e6dEkpdwwtCvH6ds1CwJT2Hdmkv
To: /content/HPV2025.xlsx
100%|██████████| 66.0k/66.0k [00:00<00:00, 57.0MB/s]


Dataset shape: (588, 12)

Train class distribution:
HPV Status
1.0    424
0.0     46
Name: count, dtype: int64

Test class distribution:
HPV Status
1.0    106
0.0     12
Name: count, dtype: int64

Example patient text:
This patient is a 69-year-old male. The patient is a non-smoker and consumes alcohol. Performance status is 1.0. The patient has experienced disease relapse. Relapse-free survival is 1130.0 days. Treatment type is 1.0. Tumour stage is T2. Nodal stage is N2. Metastatic stage is M0. Predict whether the patient is HPV positive or HPV negative.


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Class weights: tensor([5.1087, 0.5542], device='cuda:0')
Trainable parameters: 100226
Epoch 050, Train=0.6722, Test=0.6729, BA=0.6124, AUC=0.7374, Best Loss Epoch=50, Best BA Epoch=42, Best AUC Epoch=7
Epoch 100, Train=0.6366, Test=0.6305, BA=0.6344, AUC=0.7351, Best Loss Epoch=99, Best BA Epoch=63, Best AUC Epoch=7
Epoch 150, Train=0.5987, Test=0.6023, BA=0.6360, AUC=0.7398, Best Loss Epoch=146, Best BA Epoch=63, Best AUC Epoch=7
Epoch 200, Train=0.6219, Test=0.5852, BA=0.6910, AUC=0.7508, Best Loss Epoch=195, Best BA Epoch=63, Best AUC Epoch=199
Epoch 250, Train=0.5849, Test=0.5671, BA=0.6918, AUC=0.7657, Best Loss Epoch=247, Best BA Epoch=238, Best AUC Epoch=250
Epoch 300, Train=0.5815, Test=0.5544, BA=0.6863, AUC=0.7783, Best Loss Epoch=296, Best BA Epoch=290, Best AUC Epoch=289
Epoch 350, Train=0.5605, Test=0.5471, BA=0.7429, AUC=0.7814, Best Loss Epoch=347, Best BA Epoch=290, Best AUC Epoch=349
Epoch 400, Train=0.5552, Test=0.5380, BA=0.7335, AUC=0.7885, Best Loss Epoch=400, Bes